<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/scripts/early_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 2: Early eval smoke test

Not the real eval loop -- just confirms
[Qwen2-Audio-7B-Instruct](https://huggingface.co/Qwen/Qwen2-Audio-7B-Instruct)
can consume one of our hand-built `examples.json` entries end to end, and
checks that:

1. the number of encoded audio frames scales with the clip's duration (a 5s
   clip shouldn't collapse to the same token budget as a 1s clip)
2. every input tensor has the shape we expect

**Requires a GPU** (Runtime > Change runtime type > GPU). Run
`scripts/load_datasets.ipynb` first so `examples.json`'s audio paths exist.

## Setup

In [ ]:
import os

# Land at the repo root regardless of how this notebook was opened:
#  - fresh Colab (via the badge above): only this notebook exists yet -> clone
#  - local Jupyter opened from inside scripts/ -> just cd up one level
#  - local Jupyter opened from the repo root -> already there, no-op
if os.path.exists("requirements.txt"):
    pass
elif os.path.exists("../requirements.txt"):
    %cd ..
else:
    !git clone -q https://github.com/AdvayRoongta/audio-initial-dataset.git repo
    %cd repo

!pip install -q -r requirements.txt


## Pick an example

In [ ]:
import json

EXAMPLE_INDEX = 0

with open("examples.json") as f:
    examples = json.load(f)

example = examples[EXAMPLE_INDEX]
print(f"example id: {example['id']}  ({example['conflict_type']})")
print(f"audio_path: {example['audio_path']}")
print(f"text_prompt: {example['text_prompt']!r}")


## Load the model

First run downloads ~17 GB.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.float16
)
model.eval()
print("model loaded:", model.dtype, model.device)


## Run the example through the model

In [ ]:
import librosa

audio, sr = librosa.load(example["audio_path"], sr=processor.feature_extractor.sampling_rate)
duration_sec = len(audio) / sr

conversation = [{
    "role": "user",
    "content": [
        {"type": "audio", "audio_url": example["audio_path"]},
        {"type": "text", "text": example["text_prompt"]},
    ],
}]
text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(text=[text_prompt], audios=[audio], sampling_rate=sr, return_tensors="pt")
inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

print("INPUT TENSOR SHAPES")
for k, v in inputs.items():
    if hasattr(v, "shape"):
        print(f"  {k:22s} {tuple(v.shape)}")

# `feature_attention_mask` marks which of the (padded/truncated) audio
# frames from the Whisper-style feature extractor are real -- summing it
# gives the actual encoded frame count, which should track audio duration.
print("\nAUDIO DURATION vs. ENCODED FRAMES")
print(f"  clip duration         {duration_sec:.2f}s")
if "feature_attention_mask" in inputs:
    n_frames = int(inputs["feature_attention_mask"].sum())
    print(f"  valid feature frames  {n_frames}  ({n_frames / duration_sec:.1f} frames/sec)")
print(f"  input_ids length      {inputs['input_ids'].shape[-1]} tokens")


## Generate + compare to ground truth

In [ ]:
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=32)
reply = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

print(f"model reply:        {reply!r}")
print(f"ground_truth_audio: {example['ground_truth_audio']!r}")
print(f"ground_truth_text:  {example['ground_truth_text']!r}")
